# Score the frozen test sets and export paper results

Run after all 75 student fits and their audits have completed. This checks the entire matrix before evaluating the final cohorts, publishes per-image logits, predictions and confusion matrices and writes accuracy.csv, paired.json, resources.csv and reuse.csv to the Kaggle output folder.

**Setup:** GPU T4 ×2, Internet enabled, all three project inputs attached, and the Kaggle secret `HF_TOKEN` enabled for this notebook. Keep the token out of code. The repository and worker settings are filled in. Use **Run All**. The mounted files and split are pinned by their hashes. See KAGGLE_RUN_INSTRUCTIONS.md for links and recovery.

In [ ]:
HF_REPO = "Shanmuk4622/kd-crop-budget-4w-2026"
TINY_KAGGLE_SOURCE = "shanmuk4622/tinyimagenet200"
GROUP = "all"
NUM_WORKERS = 4
VERSIONS = {}  # File hashes pin each Kaggle input automatically
INPUT_PATHS = {}  # Usually detected. To override: {"cifar100": "/kaggle/input/.../dataset-cifar100-python"}
SCRATCH = "/kaggle/temp/kd_view_v2"  # Can also use /kaggle/working/kd_view_v2

### Runtime dependencies

Installs missing Python packages only. Image data are read exclusively from Kaggle mounts.

In [ ]:
import importlib.util, subprocess, sys
packages = {"huggingface_hub":"huggingface_hub", "timm":"timm", "pynvml":"nvidia-ml-py", "scipy":"scipy", "filelock":"filelock"}
missing = [package for module,package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--disable-pip-version-check",*missing])

### Embedded experiment code

The self-contained bundle verifies its digest before import.

In [ ]:
import base64, hashlib, io, os, sys, tempfile, zipfile
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "44b34996bf45cc167a213a61467b66d268a2c0062c61c259cd94f11fc573c2f9"
runtime = Path(tempfile.gettempdir()) / "kd-runtime-44b34996bf45cc"
runtime.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
    for name in bundle.namelist():
        dest = runtime / name
        value = bundle.read(name)
        if dest.exists() and dest.read_bytes() != value:
            raise RuntimeError("Incompatible extracted runtime. Restart in a clean session.")
        if not dest.exists(): dest.write_bytes(value)
sys.path.insert(0,str(runtime))
print("Verified experiment runtime", "44b34996bf45cc")

### Locate Tiny ImageNet

Select the dataset folder inside the Kaggle mount before qualification, training or scoring.

In [ ]:
"""Resolve the Tiny ImageNet mount before entering the published experiment runtime."""

from pathlib import Path


def tiny_input_path(source, override=None, base=Path("/kaggle/input")):
    owner, slug = source.split("/")
    if not owner or not slug or owner in (".", "..") or slug in (".", ".."):
        raise ValueError("Set TINY_KAGGLE_SOURCE to the attached Kaggle dataset's owner/slug.")
    base = base.resolve()
    if override:
        root = Path(override).resolve()
        if not root.is_dir() or not root.is_relative_to(base) or slug not in root.parts:
            raise ValueError(f"Set INPUT_PATHS['tiny_imagenet200'] to the mounted folder for {source}.")
    else:
        owned = base / "datasets" / owner / slug
        roots = [p for p in (owned, base / slug) if p.is_dir()]
        if not roots:
            roots = [p for p in base.rglob(slug) if p.is_dir()]
        if len(roots) != 1:
            raise ValueError(f"Attach {source} or specify its exact folder in INPUT_PATHS. Found {roots}.")
        root = roots[0]

    # Check the canonical layout before walking a mount with 120,000 images.
    for folder in (root / "tiny-imagenet-200", root):
        if ((folder / "wnids.txt").is_file() and (folder / "train").is_dir()
                and (folder / "val").is_dir() and not (folder / "tiny-imagenet-200").is_dir()):
            return str(folder)
    markers = [p for p in root.rglob("wnids.txt") if p.is_file() and "__MACOSX" not in p.parts]
    if not markers:
        # The experiment reader also supports the original, unopened Stanford ZIP.
        return str(root)

    candidates = []
    for marker in markers:
        folder = marker.parent
        # The reader must see a single class list beneath the selected folder.
        if any(other != marker and other.is_relative_to(folder) for other in markers):
            continue
        if (folder / "train").is_dir() and (folder / "val").is_dir():
            candidates.append(folder)
    if len(candidates) != 1:
        raise ValueError(
            f"Found {len(candidates)} self-contained Tiny ImageNet folders in {root}: {candidates}. "
            "Select one complete folder in INPUT_PATHS['tiny_imagenet200'], or attach the original Stanford ZIP."
        )
    return str(candidates[0])


from kaggle_inputs import SOURCES

SOURCES["tiny_imagenet200"] = TINY_KAGGLE_SOURCE
RESOLVED_PATHS = dict(INPUT_PATHS)
RESOLVED_PATHS["tiny_imagenet200"] = tiny_input_path(TINY_KAGGLE_SOURCE, INPUT_PATHS.get("tiny_imagenet200"))
print("Tiny ImageNet input:", RESOLVED_PATHS["tiny_imagenet200"])


### Execute this stage

In [ ]:
from kaggle_run import check_settings, session, qualify, train
from kaggle_results import release
check_settings(HF_REPO, RESOLVED_PATHS, VERSIONS)
with session(HF_REPO, SCRATCH, 'results') as (hub, base, output):
    result = release(hub,base,output,RESOLVED_PATHS,VERSIONS)
print(result)